# STEP 2: 移動と周辺表示 / Movement and neighborhood view — 模範解答 / Model answers

解答の一例です。元の問題文と確認セルを残し、練習セルにも解答と説明を記載しています。


## 使い方 / Workflow

このノートブック内で関数を実装し、結果を予想してから確認セルを実行します。
Implement functions directly in this notebook. Predict results before running the check cells.

1. 関数のセルを編集して実行する。 / Edit and run the function cell.
2. 確認セルを実行し、予想・実際・理由を記録する。 / Run the checks and record predictions, results and reasons.
3. 完成した関数を指定の `.py` へコピーする。 / Copy completed functions into the specified `.py` files.
4. `main.py` を端末から実行して全体を確認する。 / Run `main.py` in a terminal to check the integrated program.

関数を書き直したら定義セルを再実行してください。最後にカーネルを再起動し、上から順に実行します。
Rerun the definition cell after editing a function. Finally restart the kernel and run all cells from the top.

未実装のセルでは確認が失敗します。確認を消すのではなく、実装を見直してください。
Checks fail until the functions are implemented. Revisit the implementation rather than deleting checks.

Pythonカーネルが必要です。ノートブックは練習場所、`.py` はゲームとして動かす完成品です。
A Python kernel is required. The notebook is your practice space; the `.py` files form the working game.

## 基本課題 / Core exercise

STEP 1〜4が第13回までの必須実装範囲です。例のデータはゲーム本体と独立しているため、失敗しても作り直せます。
STEPs 1–4 are required by lesson 13. Sample data is independent of the game and can be recreated after a failed experiment.

## 1. 方向 / Directions

WASDを座標変化へ変換します。
Convert WASD to coordinate changes.

In [1]:
def get_move(command):
    moves = {"w": (0, -1), "a": (-1, 0), "s": (0, 1), "d": (1, 0)}
    return moves.get(command)


### 予想してから確認 / Predict, then check

In [2]:
for command, expected in {"w": (0, -1), "a": (-1, 0), "s": (0, 1), "d": (1, 0), "xyz": None}.items():
    assert get_move(command) == expected

### `.py`へ移す / Transfer to `.py`

`exercises/player.py` の get_move へコピー。
Copy to get_move in exercises/player.py.

In [3]:
m = [list("#####"), list("#...#"), list("#.B.#"), list("#...#"), list("#####")]

### セクション練習 / Section exercise

(3, 4)からw/a/s/dを順に使った位置を予想して確認し、不正な文字も試してください。 / Predict and check positions after w/a/s/d from (3, 4), and try an invalid character.

実行前の予想と確認後の説明も書いてください。 / Record your prediction and explain the result.


In [4]:
x, y = 3, 4
for command in ["w", "a", "s", "d", "?"]:
    movement = get_move(command)
    if movement is not None:
        dx, dy = movement
        x, y = x + dx, y + dy
    print(command, movement, (x, y))
assert (x, y) == (3, 4)


w (0, -1) (3, 3)
a (-1, 0) (2, 3)
s (0, 1) (2, 4)
d (1, 0) (3, 4)
? None (3, 4)


**予想 / Prediction:** 順に(3, 3)、(2, 3)、(2, 4)、(3, 4)。不正な文字はNoneを返すので座標を変更しない。ここでは方向変換のみを確認し、壁判定は次節で扱う。

**結果・説明 / Result and explanation:** 順に(3, 3)、(2, 3)、(2, 4)、(3, 4)。不正な文字はNoneを返すので座標を変更しない。ここでは方向変換のみを確認し、壁判定は次節で扱う。


## 2. 壁と境界 / Walls and bounds

範囲外と壁を拒否し、施設には入れるようにします。
Reject out-of-bounds cells and walls, but allow facilities.

In [5]:
def can_move(casino_map, x, y):
    if not (0 <= y < len(casino_map)):
        return False
    if not (0 <= x < len(casino_map[y])):
        return False
    return casino_map[y][x] != "#"


### 予想してから確認 / Predict, then check

In [6]:
assert can_move(m, 1, 1) is True
assert can_move(m, 2, 2) is True
for x, y in [(0, 0), (-1, 1), (1, -1), (5, 1), (1, 5)]:
    assert can_move(m, x, y) is False

### `.py`へ移す / Transfer to `.py`

`exercises/player.py` の can_move へコピー。
Copy to can_move in exercises/player.py.

### セクション練習 / Section exercise

自作の小さなマップで通路・壁・施設・負の座標・右端の範囲外を確認してください。 / On your own small map, check a corridor, wall, facility, negative coordinate and position beyond the right edge.

実行前の予想と確認後の説明も書いてください。 / Record your prediction and explain the result.


In [7]:
small_map = [list("####"), list("#.C#"), list("####")]
for x, y, expected in [(1, 1, True), (0, 1, False), (2, 1, True),
                        (-1, 1, False), (4, 1, False)]:
    actual = can_move(small_map, x, y)
    assert actual is expected
    print((x, y), actual)


(1, 1) True
(0, 1) False
(2, 1) True
(-1, 1) False
(4, 1) False


**予想 / Prediction:** 通路と施設はTrue、壁・負の座標・右端の範囲外はFalse。添字で参照する前に範囲を調べる。

**結果・説明 / Result and explanation:** 通路と施設はTrue、壁・負の座標・右端の範囲外はFalse。添字で参照する前に範囲を調べる。


## 3. 5×5を抽出 / Extract a 5×5 view

中央をプレイヤー位置とし、範囲外を # で埋めます。
Center the view on the player and fill outside cells with #.

In [8]:
def extract_view(casino_map, player):
    view = []
    for y in range(player["y"] - 2, player["y"] + 3):
        row = []
        for x in range(player["x"] - 2, player["x"] + 3):
            if 0 <= y < len(casino_map) and 0 <= x < len(casino_map[y]):
                row.append(casino_map[y][x])
            else:
                row.append("#")
        view.append(row)
    return view


### 予想してから確認 / Predict, then check

In [9]:
from copy import deepcopy
before = deepcopy(m)
for x, y in [(2, 2), (1, 1), (0, 0), (4, 4)]:
    view = extract_view(m, {"x": x, "y": y})
    assert len(view) == 5 and all(len(row) == 5 for row in view)
    for row in range(5):
        for col in range(5):
            mx, my = x + col - 2, y + row - 2
            expected = m[my][mx] if 0 <= my < len(m) and 0 <= mx < len(m[my]) else "#"
            assert view[row][col] == expected
    view[2][2] = "@"
assert m == before

### `.py`へ移す / Transfer to `.py`

`exercises/map_view.py` へコピーします。既存の show_map は中央 [2][2] を @ にして表示します。
Copy into exercises/map_view.py. The existing show_map displays @ at [2][2].

In [10]:
def move_player(casino_map, player, movement):
    dx, dy = movement
    next_x = player["x"] + dx
    next_y = player["y"] + dy
    if (dx == 0 and dy == 0) or not can_move(casino_map, next_x, next_y):
        return False
    player["x"] = next_x
    player["y"] = next_y
    player["turn"] += 1
    return True

In [11]:
p = {"x": 1, "y": 1, "turn": 0}
assert move_player(m, p, (0, -1)) is False
assert p == {"x": 1, "y": 1, "turn": 0}
assert move_player(m, p, (1, 0)) is True
assert p == {"x": 2, "y": 1, "turn": 1}

### セクション練習 / Section exercise

左上と右下の5×5を紙に描いてから関数の表示結果と比べ、中央の元の記号も確認してください。 / Draw expected 5×5 views at the top-left and bottom-right, compare with the function results and check the original center symbol.

実行前の予想と確認後の説明も書いてください。 / Record your prediction and explain the result.


In [12]:
expected_views = {
    (0, 0): ["#####", "#####", "#####", "###..", "###.B"],
    (4, 4): ["B.###", "..###", "#####", "#####", "#####"],
}
for (x, y), expected in expected_views.items():
    view = extract_view(m, {"x": x, "y": y})
    assert ["".join(row) for row in view] == expected
    assert view[2][2] == m[y][x] == "#"
    print((x, y))
    print("\n".join("".join(row) for row in view))


(0, 0)
#####
#####
#####
###..
###.B
(4, 4)
B.###
..###
#####
#####
#####


**予想 / Prediction:** 左上と右下では範囲外が#で埋まる。中央は元の記号#であり、extract_view自体は@を重ねない。予想図はexpected_viewsに記載した。

**結果・説明 / Result and explanation:** 左上と右下では範囲外が#で埋まる。中央は元の記号#であり、extract_view自体は@を重ねない。予想図はexpected_viewsに記載した。
